In [ ]:
# Hybrid: Selenium + OCR to extract U.S. Scott Catalog into CSV
# --------------------------------------------------------------
# Requirements:
#   pip install selenium pandas easyocr pillow webdriver-manager
#   Install Tesseract OCR if you want to use pytesseract instead
# --------------------------------------------------------------

# --- INSTALL DEPENDENCIES ---
!pip install easyocr
!pip install selenium pandas webdriver-manager pillow numpy

# --- INSTALL CHROME AND CHROMEDRIVER ---
!apt-get update
!apt-get install -y software-properties-common apt-transport-https wget ca-certificates gnupg2
!wget -q -O - https://dl.google.com/linux/linux_signing_key.pub | gpg --dearmor | tee /etc/apt/keyrings/google-chrome.gpg > /dev/null
!echo "deb [arch=amd64 signed-by=/etc/apt/keyrings/google-chrome.gpg] http://dl.google.com/linux/chrome/deb/ stable main" | tee /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install -y google-chrome-stable

import time
import re
import pandas as pd
import easyocr
from PIL import Image
from io import BytesIO
import numpy as np

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- CONFIG ---
USERNAME = "swgood@uci.edu"   # replace with your subscription email
PASSWORD = "d@asc1ence2025"
LOGIN_URL = "https://www.amosadvantage.com/Users/Account/LogOn"
POST_LOGIN_URL = "https://www.amosadvantage.com/"
DIGITAL_EDITIONS_URL = "https://www.amosadvantage.com/digital-editions"
OUTPUT_CSV = "us_stamps.csv"

# --- OCR setup ---
reader = easyocr.Reader(['en'])

# --- SETUP SELENIUM ---
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-extensions")
options.add_argument("--disable-features=VizDisplayCompositor")

driver_path = ChromeDriverManager().install()
chrome_service = Service(driver_path)
driver = webdriver.Chrome(service=chrome_service, options=options)
wait = WebDriverWait(driver, 20)

# --- LOGIN ---
driver.get(LOGIN_URL)
try:
    print("🔑 Logging in...")
    email_field = wait.until(EC.presence_of_element_located((By.NAME, "userNameOrEmail")))
    email_field.send_keys(USERNAME)

    password_field = driver.find_element(By.NAME, "password")
    password_field.send_keys(PASSWORD)

    login_button = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@type='submit' and text()='Login']")))
    login_button.click()

    time.sleep(8)
    print(f"✅ Current URL after login: {driver.current_url}")

    if driver.current_url.startswith(LOGIN_URL):
        raise Exception("Login failed. Still on login page.")

except Exception as e:
    print(f"❌ Login error: {e}")
    driver.quit()
    raise

# --- NAVIGATE INTO DIGITAL EDITIONS AND U.S. CATALOG ---
try:
    wait.until(EC.url_to_be(POST_LOGIN_URL))
    print("✅ Logged in and reached account home.")

    # Step 1: Digital Editions page
    driver.get(DIGITAL_EDITIONS_URL)
    print("➡️ Opened Digital Editions page")
    time.sleep(5)

    # Get the original window handle
    original_window = driver.current_window_handle

    # Step 2: Click "CLICK HERE TO ACCESS THE DIGITAL VERSION"
    print("Attempting to find and click the link to the Scott Reader using JavaScript...")
    scott_reader_link = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//a[contains(text(),'CLICK HERE TO ACCESS THE DIGITAL VERSION')]"))
    )
    driver.execute_script("arguments[0].click();", scott_reader_link)
    print("🖱️ Clicked Digital Version link")

    # Wait for a new window or tab to open and switch to it
    wait.until(EC.number_of_windows_to_be(2))
    for window_handle in driver.window_handles:
        if window_handle != original_window:
            driver.switch_to.window(window_handle)
            break
    print("✅ Switched to new window/tab.")

    # Wait for the URL to change to the reader domain (default.aspx with dynamic token) in the new window
    wait.until(EC.url_contains("reader.scottonline.com/default.aspx"))
    print(f"✅ Inside Scott Reader: {driver.current_url}")
    time.sleep(10)

    # Step 3: Click "United States"
    print("Attempting to find and click the United States catalog link...")
    # Use a more specific locator based on the HTML provided: //a[@href='https://reader.scottonline.com/volume1/unitedstates/index.html']
    us_catalog_element = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//a[@href='https://reader.scottonline.com/volume1/unitedstates/index.html']"))
    )
    # Use JavaScript click again for robustness
    driver.execute_script("arguments[0].click();", us_catalog_element)
    print("🖱️ Clicked United States catalog link")

    # Wait for the URL to change to indicate the US catalog viewer is loaded
    # This might be a different URL than the index.html one, so we'll wait for
    # a URL containing "volume1/unitedstates" as a general indicator.
    wait.until(EC.url_contains("volume1/unitedstates"))
    print(f"✅ U.S. catalog is open: {driver.current_url}")
    time.sleep(10)

except Exception as e:
    print(f"❌ Navigation error: {e}")
    driver.quit()
    raise

# --- SCRAPE PAGES WITH OCR ---
stamps = []
page_num = 1

while True:
    try:
        print(f"📄 Processing page {page_num}...")

        # Print the current URL at the beginning of each page processing iteration
        print(f"  Current URL for page {page_num}: {driver.current_url}")

        # Add explicit wait for the page content to be present after potential navigation
        # This might need to be adjusted based on elements that are present on the page after loading
        # We need a reliable element that appears when a catalog page is loaded.
        # For now, keeping body as a fallback, but a more specific element is better.
        # Let's try waiting for a common element within the viewer, like a canvas or image
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "canvas, img, body")))
        print("  Page content loaded.")


        # Capture screenshot
        png = driver.get_screenshot_as_png()
        img = Image.open(BytesIO(png))

        # Convert PIL Image to numpy array
        img_np = np.array(img)

        # OCR
        # Add a check for empty lines before processing
        results = reader.readtext(img_np, detail=0)
        lines = [line.strip() for line in results if line.strip()]

        if not lines:
            print(f"  No text found on page {page_num}. This might be an empty page or the end.")
            # If no text is found, try to navigate to the next page anyway,
            # as some pages might be blank or contain only images without text.
            # If we consistently find no text and fail to navigate, we'll break later.


        # Split into chunks by Scott number
        chunks, current_chunk = [], []
        for line in lines:
            # Assuming Scott numbers are typically 3 or 4 digits, optionally followed by a letter
            # Refined regex to be more flexible with leading/trailing spaces and punctuation
            if re.match(r"^\s*\d{3,4}[A-Z]?[\s.,;:]", line):
                if current_chunk:
                    chunks.append(current_chunk)
                current_chunk = [line]
            else:
                if current_chunk:
                    current_chunk.append(line)
        if current_chunk:
            chunks.append(current_chunk)

        parsed_entries = []
        for chunk in chunks:
            text = " ".join(chunk).replace(",", ".")
            # Use the refined regex for matching Scott number
            scott_match = re.search(r"\b(\d{3,4}[A-Z]?)\b", text)
            # Refined year pattern to look for 4-digit years, potentially at the start or within text
            year_match = re.search(r"(?:^|\s)(18\d{2}|19\d{2}|20\d{2})(?:$|\s)", text)
            # Refined price pattern to be more robust, looking for numbers with exactly two decimal places
            prices = re.findall(r"\b\d+\.\d{2}\b", text)

            # Attempt to extract description more accurately
            description = text
            if scott_match:
                # Remove the matched Scott number from the description
                description = re.sub(re.escape(scott_match.group(0)), "", description, 1).strip()
            if year_match:
                 # Remove the matched year from the description
                 description = re.sub(re.escape(year_match.group(0)), "", description, 1).strip()

            # Simple assignment for now, assuming the first two found prices are Mint and Used
            mint_value = prices[0] if len(prices) > 0 else None
            used_value = prices[1] if len(prices) > 1 else None


            parsed_entries.append({
                "Scott Number": scott_match.group(1) if scott_match else None,
                "Year": year_match.group(1) if year_match else None,
                "Description": description,
                "Mint Value": mint_value,
                "Used Value": used_value,
                "Source Page": page_num
            })

        # If no entries were parsed from the page, and it's not the first page,
        # it might indicate the end or an issue.
        if not parsed_entries and page_num > 1:
             print(f"No entries found on page {page_num} and it's not the first page. Assuming end of catalog.")
             break

        # If it's the first page and no entries are found, it could be an introductory page.
        # We should still try to navigate to the next page.


        stamps.extend(parsed_entries)


        print(f"Processed page {page_num}. Found {len(parsed_entries)} entries.")

        # --- NAVIGATE TO NEXT PAGE ---
        # Attempt to find and click the next page button within the reader interface.
        # This locator might need to be adjusted based on the actual HTML of the reader.
        try:
            # Assuming the next page button has the class "nextPage" based on previous attempt error
            next_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".nextPage"))) # Using the previously attempted selector
            next_btn.click()
            print("Clicked next page button.")
            time.sleep(5) # Add a delay to allow the next page to load
            page_num += 1 # Increment page number only if navigation was attempted
        except Exception as e:
            print("Could not find or click next page button. Assuming end of catalog or navigation issue.")
            break # Exit loop if next button is not found or clickable


        # Set a limit for the number of pages to process to avoid infinite loops during testing
        # Remove or adjust this limit for full catalog processing
        if page_num > 10: # Process first 10 pages for now
            print("⏹️ Stopping at 10 pages (demo mode).")
            break


    except Exception as e:
        print(f"❌ An error occurred while processing page {page_num}: {e}")
        # If any other error occurs during processing, break the loop
        break

# --- SAVE TO CSV ---
df = pd.DataFrame(stamps)
df.to_csv(OUTPUT_CSV, index=False)

print(f"✅ Saved {len(df)} entries into {OUTPUT_CSV}")

# --- CLEANUP ---
driver.quit()

try:
    from google.colab import files
    files.download("us_stamps.csv")
except ImportError:
    pass